# 03 — Train / validation / test split

**Decision:** use a chronological 70% / 15% / 15% split by purchase time. Deployment
predicts future orders, so later validation/test periods are a more realistic check for
drift than random shuffling. The split happens before detailed EDA. We inspect only date
ranges, sizes, and label balance here.

**Reads:** labeled table. **Writes:** train, validation, test CSV files and split summary.

In [25]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.py").exists(): ROOT = ROOT.parent
SOURCE = ROOT / "artifacts" / "02_labels" / "labeled_orders.csv"
OUT = ROOT / "artifacts" / "03_split"
OUT.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists(), "Run notebook 02 first"

data = pd.read_csv(SOURCE, parse_dates=["order_purchase_timestamp"])
data = data.sort_values(["order_purchase_timestamp", "order_id"]).reset_index(drop=True)
train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)
splits = {
    "train": data.iloc[:train_end].copy(),
    "validation": data.iloc[train_end:validation_end].copy(),
    "test": data.iloc[validation_end:].copy(),
}
assert sum(map(len, splits.values())) == len(data)
assert set(splits["train"].order_id).isdisjoint(splits["validation"].order_id)
assert set(splits["train"].order_id).isdisjoint(splits["test"].order_id)
assert splits["train"].order_purchase_timestamp.max() <= splits["validation"].order_purchase_timestamp.min()
assert splits["validation"].order_purchase_timestamp.max() <= splits["test"].order_purchase_timestamp.min()

In [26]:
summary = []
for name, frame in splits.items():
    frame.to_csv(OUT / f"{name}.csv", index=False)
    summary.append({
        "split": name, "rows": len(frame), "share": len(frame) / len(data),
        "start": frame["order_purchase_timestamp"].min().isoformat(),
        "end": frame["order_purchase_timestamp"].max().isoformat(),
        "late_count": int(frame["is_late"].sum()),
        "late_share": float(frame["is_late"].mean()),
    })
summary = pd.DataFrame(summary)
summary.to_csv(OUT / "split_summary.csv", index=False)
display(summary)
print("Chronological balance may differ across periods; that is useful drift information, not a reason to leak future data.")

,split,rows,share,start,end,late_count,late_share
0,train,67529,0.700000,2016-09-15T12:16:38,2018-04-15T20:12:35,6096,0.090272
1,validation,14470,0.149995,2018-04-15T20:17:11,2018-06-21T08:29:29,773,0.053421
2,test,14471,0.150005,2018-06-21T08:41:07,2018-08-29T15:00:37,957,0.066132


Chronological balance may differ across periods; that is useful drift information, not a reason to leak future data.
